# ARCH 6133 Places / Platforms
## Street View Lab: Reading the Street as Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danmillr/places-platforms/blob/main/tutorials/ARCH6133_Street_View_Lab.ipynb)

---

### What this notebook does

This is a **field guide for the screen**. You will walk a block of New York City through Google Street View, gathering the same kinds of evidence a designer collects on foot: how enclosed the street feels, what fills the sidewalk, who the signs are talking to, how the block has changed across the last decade.

You will end with a small structured dataset — one row per Street View image — that joins:
- a historical timeline of the same location,
- a spatial sample of a full street segment,
- a depth reading of perceived enclosure,
- a vanishing-point reading of perspective geometry,
- a semantic segmentation of the visible scene,
- an OCR reading of the street's signage ecology.

The notebook is organized into nine modules. Run them in order the first time. Each module saves its outputs to your Google Drive, so later modules can read what earlier ones produced even if your runtime resets.

### How this is organized in the course repo

This notebook lives in the `tutorials/` folder of the [places-platforms](https://github.com/danmillr/places-platforms) repository, alongside `ARCH6133_POI_Data.ipynb`. Files in `tutorials/` follow the same convention: `ARCH6133_` prefix, a descriptive title in `Title_Case`, `.ipynb` extension, and a working **Open in Colab** badge in the first cell that points to `main`.

### Getting a Google Cloud API key

1. Go to [Google Cloud Console](https://console.cloud.google.com/).
2. Create a new project (or pick an existing one).
3. Open **APIs & Services → Library** and enable these APIs:
   - **Street View Static API**
   - **Geocoding API**
4. Open **APIs & Services → Credentials**, click **Create credentials → API key**, and copy the key.
5. Click **Restrict key** → under **API restrictions**, allow only the two APIs above. This prevents accidental misuse if your key ever leaks.

### Estimated cost

The Street View Static API is billed at roughly **\$7 per 1,000 images** (rates may shift; check the [current pricing](https://developers.google.com/maps/documentation/streetview/usage-and-billing)). Google currently grants every billing account a recurring monthly credit, which usually covers everything you will do in this lab — but **set a hard cap anyway**.

To set a billing cap:
1. In Google Cloud Console, open **Billing → Budgets & alerts**.
2. Create a budget of **\$10/month** with email alerts at 50%, 90%, and 100%.
3. This will not auto-stop API calls, but it will warn you well before you spend real money.

This lab will pull roughly 25 to 60 images total per full run, depending on segment length and how many historical years are available. Stay under your cap easily.

### A note on what we are doing here

Street View is the most heavily mediated way of "seeing" a city ever built. Every pixel has been chosen, stitched, blurred, and time-stamped by a single private company. We are not pretending these images are the street — we are studying what this particular representational platform makes visible, and what it leaves out.


---

## Module 0 — Setup and Configuration

We will mount your Google Drive, install everything we need in a single command, pull your API key out of Colab's secret store, and pick the address we will study.

If your runtime ever resets (it eventually will), come back to the top of the notebook and re-run from here. Drive mounts, package installs, and model weights do not persist between sessions — only the files you saved to Drive do.


### Mount Google Drive

Everything this notebook saves — images, CSVs, model weights, GeoJSON — goes into a folder inside your Drive. Run the cell below and accept the permission dialog when Colab asks. The mount point `/content/drive/MyDrive/` is just your Drive's root, viewed from inside this virtual machine.

In [ ]:
# Mount your personal Google Drive into the Colab filesystem.
# After this runs, /content/drive/MyDrive/ behaves like a normal folder.
from google.colab import drive
drive.mount('/content/drive')

### Install all required packages

We install everything in one shot so you never have to remember which library belongs to which module. This cell takes about a minute the first time and will print a lot of dependency-resolution noise — that is normal.

In [ ]:
# One consolidated install. Run once per session.
# Quiet flag (-q) cuts noise; everything we use is already on Colab except
# easyocr, so this is mostly fetching that one wheel and its deps.
!pip install -q easyocr requests Pillow numpy opencv-python matplotlib pandas scikit-image tqdm torchvision

### Store your API key in Colab Secrets

Before running this notebook, you need to store your Google API key in Colab Secrets so it never appears in your code or gets accidentally shared.

To do this:
1. Click the key icon in the left sidebar (Secrets)
2. Click 'Add new secret'
3. Set the Name to: `GOOGLE_API_KEY`
4. Paste your API key as the Value
5. Toggle 'Notebook access' to ON for this notebook
6. Return here and run this cell

Your key is now stored securely and will not appear anywhere in the notebook code.

In [ ]:
# Pull the API key out of Colab's secret store.
# It never appears in this notebook, in your Drive, or in any saved output.
from google.colab import userdata

API_KEY = userdata.get('GOOGLE_API_KEY')

# Fail loudly with a plain-English message if the key isn't there.
if not API_KEY:
    raise ValueError(
        "API key not found. Please add GOOGLE_API_KEY to Colab Secrets "
        "and enable notebook access. See the instructions above."
    )
else:
    print("API key loaded successfully. You are ready to go.")

### Configure the study location

Pick the address you want to study and where outputs should land in your Drive. The default is **125th Street and Lexington Avenue in East Harlem**, the spine of our course site. You can swap in any street address you like.

`OUTPUT_FOLDER` is a path inside your Drive. The notebook creates it (and a handful of subfolders) the first time you run Module 0.

In [ ]:
# Edit these two values, then run the cell.
TARGET_ADDRESS = "125th Street and Lexington Avenue, New York, NY"
OUTPUT_FOLDER  = "/content/drive/MyDrive/StreetViewLab/"

# We import here so this cell stands alone if you re-run it.
import os

# Create the project folder and one subfolder per module that writes files.
SUBFOLDERS = ["images", "timeline", "segment", "centroids", "ocr", "models"]
for sub in [""] + SUBFOLDERS:
    os.makedirs(os.path.join(OUTPUT_FOLDER, sub), exist_ok=True)

print(f"Output folder ready: {OUTPUT_FOLDER}")
for sub in SUBFOLDERS:
    print(f"  - {sub}/")

### Geocoding helper

A geocoder turns a plain-text address into a latitude / longitude pair. We will use the Google Geocoding API. The function below is the one piece of plumbing every other module depends on, so we wrap it once and reuse it everywhere.

The function returns `None` on failure rather than crashing the notebook — read the printed message to debug.

In [ ]:
# Geocoding helper. Takes a plain-text address, returns (lat, lng) or None.
import requests

def geocode_address(address, api_key):
    """Convert a street address to (lat, lng) using Google Geocoding."""
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": address, "key": api_key}
    try:
        response = requests.get(url, params=params, timeout=15)
        data = response.json()
    except Exception as e:
        print(f"Geocoding request failed: {e}")
        return None

    # Google reports its own status in the JSON body, not the HTTP code.
    if data.get("status") != "OK" or not data.get("results"):
        print(f"Geocoder could not resolve '{address}'. Status: {data.get('status')}")
        return None

    result = data["results"][0]
    loc = result["geometry"]["location"]
    formatted = result["formatted_address"]
    print(f"Found: {formatted} at {loc['lat']}, {loc['lng']}")
    return (loc["lat"], loc["lng"])

# Try it on our target address so we know the key works end-to-end.
TARGET_LATLNG = geocode_address(TARGET_ADDRESS, API_KEY)

---

## Module 1 — Single Capture and Metadata

Before we go anywhere, we look at one frame. One image, facing due north, square aspect, 90° field of view — the same parameters every later module will use as a baseline.

Pay attention to the printed metadata block. The `pano_id` is the unique fingerprint of the panorama Google served us; the `date` is when Google's car drove past. Both will matter when we ask, in Module 2, "what did this corner look like in 2014?"


In [ ]:
# Module 1 — fetch one Street View image at the target location.
import os
import requests
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO

# Parameters the rest of the notebook will reuse as defaults.
DEFAULT_HEADING = 0    # compass degrees; 0 = north
DEFAULT_PITCH   = 0    # camera tilt; 0 = horizon
DEFAULT_SIZE    = "640x640"
DEFAULT_FOV     = 90   # field of view in degrees

def fetch_streetview_image(lat, lng, api_key,
                           heading=DEFAULT_HEADING,
                           pitch=DEFAULT_PITCH,
                           size=DEFAULT_SIZE,
                           fov=DEFAULT_FOV):
    """Return a PIL Image from the Street View Static API, or None on failure."""
    url = "https://maps.googleapis.com/maps/api/streetview"
    params = {
        "size": size,
        "location": f"{lat},{lng}",
        "heading": heading,
        "pitch": pitch,
        "fov": fov,
        "key": api_key,
    }
    try:
        r = requests.get(url, params=params, timeout=20)
        r.raise_for_status()
        return Image.open(BytesIO(r.content)).convert("RGB")
    except Exception as e:
        print(f"Could not fetch image at {lat},{lng}: {e}")
        return None

def fetch_streetview_metadata(lat, lng, api_key):
    """Return the panorama metadata dict, or None on failure."""
    url = "https://maps.googleapis.com/maps/api/streetview/metadata"
    params = {"location": f"{lat},{lng}", "key": api_key}
    try:
        r = requests.get(url, params=params, timeout=15)
        return r.json()
    except Exception as e:
        print(f"Metadata request failed: {e}")
        return None

# Fetch one image plus its metadata.
if TARGET_LATLNG is None:
    print("Skipping Module 1 — no target coordinates. Re-run Module 0 first.")
else:
    lat, lng = TARGET_LATLNG
    img = fetch_streetview_image(lat, lng, API_KEY)
    meta = fetch_streetview_metadata(lat, lng, API_KEY)

    if img is not None:
        # Display inline, titled with the address.
        fig, ax = plt.subplots(figsize=(7, 7))
        ax.imshow(img)
        ax.set_title(TARGET_ADDRESS)
        ax.axis("off")
        plt.show()

        # Save the file to Drive.
        out_path = os.path.join(OUTPUT_FOLDER, "images", "single_capture.jpg")
        img.save(out_path, "JPEG", quality=92)
        print(f"Saved: {out_path}")

    # Print a clean metadata block.
    if meta and meta.get("status") == "OK":
        print()
        print("--- Capture metadata ---")
        print(f"Coordinates : {meta['location']['lat']}, {meta['location']['lng']}")
        print(f"Capture date: {meta.get('date', 'unknown')}")
        print(f"Panorama ID : {meta.get('pano_id', 'unknown')}")
        print(f"Heading     : {DEFAULT_HEADING} degrees")
        print(f"Image size  : {DEFAULT_SIZE}")
    else:
        print(f"Metadata unavailable. Status: {meta.get('status') if meta else 'no response'}")

---

## Module 2 — Historical Street View Timeline

Google's Street View car has driven most New York streets several times — sometimes once a year, sometimes more. The metadata API exposes the full history of panoramas captured at a given location, so we can pull one representative image per year and lay them out as a filmstrip.

A note before you run this: **the Street View metadata API returns sparse data for lower-traffic streets**. If you only see two or three years for your address, that is not a code error — it is what is available. Try a busier intersection if you want more frames.

After the filmstrip renders, scroll back through it slowly and answer the prompt at the bottom of this module.

In [ ]:
# Module 2 — historical timeline.
import os
import csv
import math
import requests
from datetime import datetime
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO
from tqdm.auto import tqdm

def fetch_timeline_panos(lat, lng, api_key, radius=50):
    """Walk a small lat/lng grid around the target to surface multiple pano IDs.
    The metadata endpoint returns the nearest pano for a given lat/lng — by
    sampling a ring of nearby points we catch panos from different years that
    Google snapped at slightly offset locations."""
    panos = {}  # pano_id -> metadata dict
    # Offsets are in degrees; ~0.0001 deg ≈ 11 m at NYC's latitude.
    offsets = [(0,0)] + [(math.cos(a)*0.0003, math.sin(a)*0.0003)
                         for a in [i*math.pi/4 for i in range(8)]]
    for dlat, dlng in offsets:
        url = "https://maps.googleapis.com/maps/api/streetview/metadata"
        params = {"location": f"{lat+dlat},{lng+dlng}",
                  "radius": radius, "key": api_key}
        try:
            data = requests.get(url, params=params, timeout=15).json()
        except Exception as e:
            print(f"Metadata probe failed at offset ({dlat},{dlng}): {e}")
            continue
        if data.get("status") == "OK" and "pano_id" in data:
            panos[data["pano_id"]] = data
    return list(panos.values())

def fetch_image_by_pano(pano_id, api_key,
                       heading=DEFAULT_HEADING,
                       pitch=DEFAULT_PITCH,
                       size=DEFAULT_SIZE,
                       fov=DEFAULT_FOV):
    """Fetch a Street View image for a specific historical pano_id."""
    url = "https://maps.googleapis.com/maps/api/streetview"
    params = {"size": size, "pano": pano_id,
              "heading": heading, "pitch": pitch, "fov": fov, "key": api_key}
    try:
        r = requests.get(url, params=params, timeout=20)
        r.raise_for_status()
        return Image.open(BytesIO(r.content)).convert("RGB")
    except Exception as e:
        print(f"Could not fetch pano {pano_id}: {e}")
        return None

if TARGET_LATLNG is None:
    print("Skipping Module 2 — no target coordinates.")
else:
    lat, lng = TARGET_LATLNG
    panos = fetch_timeline_panos(lat, lng, API_KEY)

    # Sort one pano per year — keep the earliest capture in each year so the
    # timeline reads chronologically with no duplicates.
    by_year = {}
    for p in panos:
        date_str = p.get("date")
        if not date_str:
            continue
        year = date_str[:4]
        if year not in by_year or date_str < by_year[year]["date"]:
            by_year[year] = p
    timeline = sorted(by_year.values(), key=lambda p: p["date"])

    if not timeline:
        print("No historical panoramas found. Try a busier intersection.")
    else:
        print(f"Found {len(timeline)} distinct years of Street View at this location.")

        # Fetch one image per year with a progress bar.
        frames = []
        for p in tqdm(timeline, desc="Downloading timeline"):
            img = fetch_image_by_pano(p["pano_id"], API_KEY)
            if img is not None:
                frames.append((p, img))

        # Render the filmstrip.
        n = len(frames)
        fig, axes = plt.subplots(1, n, figsize=(3*n, 3.5))
        if n == 1:
            axes = [axes]
        for ax, (p, img) in zip(axes, frames):
            ax.imshow(img)
            ax.set_title(p["date"][:4], fontsize=12)
            ax.axis("off")
        plt.tight_layout()
        timeline_path = os.path.join(OUTPUT_FOLDER, "timeline", "timeline.png")
        plt.savefig(timeline_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Saved filmstrip: {timeline_path}")

        # Write the CSV.
        csv_path = os.path.join(OUTPUT_FOLDER, "timeline", "capture_dates.csv")
        with open(csv_path, "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["year", "date", "pano_id", "lat", "lng"])
            for p, _ in frames:
                w.writerow([p["date"][:4], p["date"], p["pano_id"],
                            p["location"]["lat"], p["location"]["lng"]])
        print(f"Saved capture dates: {csv_path}")

**Reflection.** What changed? What stayed the same? Note at least two observations about the street's transformation — façades repainted, retail tenants swapped, scaffolding appearing or disappearing, traffic patterns shifting, sidewalk life thickening or thinning. Type your answer below this cell or in your field notebook.

---

## Module 3 — Extract an Entire Street Segment

Now we stop standing in one spot and walk a block. Give the notebook a `START_ADDRESS` and an `END_ADDRESS`, and it will:

1. Geocode both endpoints.
2. Interpolate `NUM_POINTS` evenly spaced samples between them.
3. Calculate the compass bearing between each consecutive pair so the camera always faces forward.
4. Pull one Street View image at each sample point.
5. Display them as a grid and save them, indexed in a CSV.

This is the dataset every later module (depth, vanishing points, segmentation, OCR) reads from. If you change the segment, re-run from here.

In [ ]:
# Module 3 — sample a street segment.
import os
import csv
import math
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Edit these three. Defaults walk one block of 125th Street.
START_ADDRESS = "125th Street and Lexington Avenue, New York, NY"
END_ADDRESS   = "125th Street and Park Avenue, New York, NY"
NUM_POINTS    = 10

def compass_bearing(lat1, lng1, lat2, lng2):
    """Initial compass bearing from point 1 to point 2, in degrees [0, 360)."""
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dlng = math.radians(lng2 - lng1)
    y = math.sin(dlng) * math.cos(phi2)
    x = math.cos(phi1)*math.sin(phi2) - math.sin(phi1)*math.cos(phi2)*math.cos(dlng)
    return (math.degrees(math.atan2(y, x)) + 360) % 360

start_ll = geocode_address(START_ADDRESS, API_KEY)
end_ll   = geocode_address(END_ADDRESS,   API_KEY)

if not start_ll or not end_ll:
    print("Could not geocode one of the endpoints. Module 3 aborted.")
else:
    # Linear interpolation between endpoints. For block-length segments this is
    # accurate enough — for longer routes you would want a Directions API path.
    lat1, lng1 = start_ll
    lat2, lng2 = end_ll
    points = [(lat1 + (lat2 - lat1)*i/(NUM_POINTS-1),
               lng1 + (lng2 - lng1)*i/(NUM_POINTS-1))
              for i in range(NUM_POINTS)]
    bearing = compass_bearing(lat1, lng1, lat2, lng2)
    print(f"Segment bearing (start → end): {bearing:.1f} degrees")

    # Fetch all images with progress.
    samples = []
    for i, (la, ln) in enumerate(tqdm(points, desc="Walking segment")):
        img = fetch_streetview_image(la, ln, API_KEY, heading=bearing)
        if img is None:
            continue
        fname = f"seg_{i:03d}_{la:.6f}_{ln:.6f}.jpg"
        out = os.path.join(OUTPUT_FOLDER, "images", fname)
        img.save(out, "JPEG", quality=92)
        samples.append({"sequence": i, "lat": la, "lng": ln,
                        "heading": bearing, "filename": fname, "image": img})

    # Display as a dynamic grid.
    n = len(samples)
    cols = 5 if n >= 5 else n
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(3*cols, 3*rows))
    axes = axes.flatten() if hasattr(axes, "flatten") else [axes]
    for ax, s in zip(axes, samples):
        ax.imshow(s["image"])
        ax.set_title(f"{s['sequence']:02d}", fontsize=10)
        ax.axis("off")
    # Hide any leftover subplots.
    for ax in axes[len(samples):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    # Write the index CSV.
    csv_path = os.path.join(OUTPUT_FOLDER, "images", "segment_index.csv")
    with open(csv_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["sequence", "lat", "lng", "heading", "filename"])
        for s in samples:
            w.writerow([s["sequence"], s["lat"], s["lng"],
                        s["heading"], s["filename"]])
    print(f"Saved segment index: {csv_path}")

---

## Module 4 — Depth Map Estimation

> **Switch to a GPU runtime before running this module.** In the menu: **Runtime → Change runtime type → T4 GPU**. The MiDaS depth model is a small neural network; on CPU it works but is much slower.

A monocular depth estimator looks at a single photograph and guesses how far each pixel is from the camera. We will use **MiDaS small**, a well-known model that is small enough to run on the free Colab GPU.

Depth maps are not metric — they tell you relative distance, not feet. But that is exactly what designers care about: is the far façade pressing in on the camera (a tight, enclosed street) or receding into the distance (an open, plaza-like condition)?

After the heatmap renders, the cell prints a single classification — `near`, `mid`, or `far` — based on the average depth in the frame. That label is rough, but it is enough to give every image a coarse enclosure tag we can join into the final dataset.

In [ ]:
# Module 4 — MiDaS depth estimation.
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Cache the model weights inside your Drive so a runtime reset does not force
# a re-download. Torch Hub picks up TORCH_HOME automatically.
os.environ["TORCH_HOME"] = os.path.join(OUTPUT_FOLDER, "models")
os.makedirs(os.environ["TORCH_HOME"], exist_ok=True)

# Skip re-loading if we already have the model in memory from a previous run.
if "midas_model" not in globals():
    print("Loading MiDaS small (first run downloads ~80 MB)...")
    midas_model = torch.hub.load("intel-isl/MiDaS", "MiDaS_small")
    midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
    midas_transform = midas_transforms.small_transform
    midas_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    midas_model.to(midas_device).eval()
    print(f"MiDaS loaded on {midas_device}.")
else:
    print("MiDaS already loaded — reusing in-memory model.")

def estimate_depth(pil_image):
    """Return a (H, W) numpy array of relative depths for a PIL image."""
    img_np = np.array(pil_image)
    batch = midas_transform(img_np).to(midas_device)
    with torch.no_grad():
        prediction = midas_model(batch)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=img_np.shape[:2],
            mode="bicubic",
            align_corners=False,
        ).squeeze()
    return prediction.cpu().numpy()

# Run on the first segment image from Module 3.
segment_dir = os.path.join(OUTPUT_FOLDER, "images")
segment_files = sorted(f for f in os.listdir(segment_dir) if f.startswith("seg_"))

if not segment_files:
    print("No segment images found. Run Module 3 first.")
else:
    first_path = os.path.join(segment_dir, segment_files[0])
    img = Image.open(first_path).convert("RGB")
    depth = estimate_depth(img)

    # Normalize for display.
    depth_vis = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)

    # Side-by-side display.
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(img)
    axes[0].set_title(f"Original — {segment_files[0]}")
    axes[0].axis("off")
    hm = axes[1].imshow(depth_vis, cmap="inferno")
    axes[1].set_title("Estimated depth (brighter = closer)")
    axes[1].axis("off")
    fig.colorbar(hm, ax=axes[1], fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

    # Classify mean depth into three coarse zones.
    # MiDaS outputs higher values for closer objects.
    mean_depth = float(depth_vis.mean())
    if mean_depth >= 0.55:
        zone = "near (dense enclosure — façades pressing in)"
    elif mean_depth >= 0.35:
        zone = "mid (active streetwall — balanced street section)"
    else:
        zone = "far (open or plaza-like — recessed or expansive)"
    print(f"\nMean normalized depth: {mean_depth:.3f}")
    print(f"Classification       : {zone}")

    # Save the heatmap.
    heatmap_path = os.path.join(OUTPUT_FOLDER, "images", "depth_map.jpg")
    plt.figure(figsize=(8, 8))
    plt.imshow(depth_vis, cmap="inferno")
    plt.axis("off")
    plt.savefig(heatmap_path, dpi=150, bbox_inches="tight", pad_inches=0)
    plt.close()
    print(f"Saved heatmap: {heatmap_path}")

**Reading the depth map as urban design.** A depth heatmap is a quick proxy for *spatial enclosure* — the felt sense of being inside a room defined by buildings. The classic measure is the **street canyon ratio**: building height divided by street width. A tight ratio (≈1:1) reads as intimate and pedestrian-scaled; a loose ratio (≈1:3 or wider) reads as open and car-oriented. Mid-range readings often map onto streets that feel safest to walk, because the street section has a defined edge but does not feel oppressive. As you build intuition, compare the `near/mid/far` label to your own embodied memory of the block.

---

## Module 5 — Centroid and Vanishing Point Extraction

A vanishing point is where parallel lines in three-dimensional space appear to converge on the picture plane. In a Street View frame facing down a street, the vanishing point is almost always near the center of the image, pulled slightly off-axis by the camera heading and the geometry of the curb.

We find it the old-fashioned way: edge detection, then a Hough transform to surface dominant lines, then a voting grid to cluster where those lines intersect. The output is a small crosshair on the original image plus a 128×128 crop centered on that point — what we will call the "centroid patch" for that frame.

In [ ]:
# Module 5 — vanishing point detection via Hough lines.
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def estimate_vanishing_point(pil_image):
    """Return (x, y) of the estimated vanishing point in image pixels."""
    img = np.array(pil_image)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    # Edge detection. The thresholds are tuned for daylight Street View frames.
    edges = cv2.Canny(gray, 50, 150)

    # Probabilistic Hough lines — returns line segments as (x1,y1,x2,y2).
    lines = cv2.HoughLinesP(edges, rho=1, theta=np.pi/180,
                            threshold=80, minLineLength=40, maxLineGap=10)
    h, w = gray.shape
    if lines is None or len(lines) < 2:
        # Fall back to image center if no lines were found.
        return (w // 2, h // 2)

    # Voting grid: extend each line to a full line in normal form
    # (cos(theta)*x + sin(theta)*y = rho), then intersect every pair and
    # vote into a coarse 32x32 grid over the image plane.
    grid_w, grid_h = 32, 32
    grid = np.zeros((grid_h, grid_w), dtype=np.int32)

    def line_params(seg):
        x1, y1, x2, y2 = seg
        # Reject near-vertical and near-horizontal lines — vanishing points
        # are voted by diagonals that cross the frame.
        dx, dy = x2 - x1, y2 - y1
        if abs(dx) < 5 and abs(dy) < 5:
            return None
        # Normal form: a*x + b*y = c.
        return (dy, -dx, dy*x1 - dx*y1)

    params = [p for p in (line_params(l[0]) for l in lines) if p is not None]
    if len(params) < 2:
        return (w // 2, h // 2)

    for i in range(len(params)):
        a1, b1, c1 = params[i]
        for j in range(i+1, len(params)):
            a2, b2, c2 = params[j]
            det = a1*b2 - a2*b1
            if abs(det) < 1e-3:
                continue
            x = (c1*b2 - c2*b1) / det
            y = (a1*c2 - a2*c1) / det
            # Only count intersections inside the image frame.
            if 0 <= x < w and 0 <= y < h:
                gx = int(x / w * grid_w)
                gy = int(y / h * grid_h)
                grid[gy, gx] += 1

    # If no intersections landed inside the frame, fall back to the center
    # rather than the (0,0) corner that argmax would return on a zero grid.
    if grid.max() == 0:
        return (w // 2, h // 2)

    # Pick the highest-voted cell, return its center in image pixels.
    gy, gx = np.unravel_index(np.argmax(grid), grid.shape)
    vx = int((gx + 0.5) * w / grid_w)
    vy = int((gy + 0.5) * h / grid_h)
    return (vx, vy)

def draw_crosshair(pil_image, point, size=20, thickness=2):
    """Return a copy of the image with a red crosshair drawn at point."""
    img = np.array(pil_image).copy()
    x, y = point
    cv2.line(img, (x-size, y), (x+size, y), (255, 0, 0), thickness)
    cv2.line(img, (x, y-size), (x, y+size), (255, 0, 0), thickness)
    return Image.fromarray(img)

def crop_centroid(pil_image, point, size=128):
    """Crop a size x size patch centered on point, clipped to the frame."""
    w, h = pil_image.size
    x, y = point
    half = size // 2
    left   = max(0, min(w - size, x - half))
    upper  = max(0, min(h - size, y - half))
    return pil_image.crop((left, upper, left + size, upper + size))

# Run on every segment image.
segment_dir = os.path.join(OUTPUT_FOLDER, "images")
segment_files = sorted(f for f in os.listdir(segment_dir) if f.startswith("seg_"))

if not segment_files:
    print("No segment images found. Run Module 3 first.")
else:
    for fname in segment_files:
        img = Image.open(os.path.join(segment_dir, fname)).convert("RGB")
        vp = estimate_vanishing_point(img)
        annotated = draw_crosshair(img, vp)
        patch = crop_centroid(img, vp)

        fig, axes = plt.subplots(1, 2, figsize=(12, 5),
                                 gridspec_kw={"width_ratios": [4, 1]})
        axes[0].imshow(annotated)
        axes[0].set_title(f"{fname} — vanishing point at {vp}")
        axes[0].axis("off")
        axes[1].imshow(patch)
        axes[1].set_title("Centroid patch (128x128)")
        axes[1].axis("off")
        plt.tight_layout()
        plt.show()

        out = os.path.join(OUTPUT_FOLDER, "centroids", fname)
        patch.save(out, "JPEG", quality=92)
    print(f"Saved {len(segment_files)} centroid patches to {OUTPUT_FOLDER}centroids/")

**Why vanishing points matter.** A vanishing point encodes the perspective geometry of the scene — and therefore the *felt* width and length of the street. A vanishing point sitting low and centered reads as a long, formal axis (think Park Avenue's median sightlines). A vanishing point pulled high or off to one side reads as a more incidental, irregular street (think a Lower East Side block where every façade rakes at a different angle). Their position is one of the most reliable cues for whether a street feels **wide, narrow, monumental, or intimate**.

---

## Module 6 — Semantic Image Segmentation

> **Switch to a GPU runtime before running this module if you have not already.** **Runtime → Change runtime type → T4 GPU**. DeepLab v3 is heavier than MiDaS and will be painfully slow on CPU.

Semantic segmentation labels every pixel of an image with a category. We use **DeepLab v3 with a ResNet-50 backbone** from `torchvision.models.segmentation`.

**A real caveat before you read the results.** The torchvision DeepLab is pretrained on a subset of COCO using the **21 Pascal VOC classes** (background, person, car, bus, motorbike, pottedplant, and a handful of others). It is **not** an urban-scene parser like models trained on Cityscapes or ADE20K. That means it can reliably find people, vehicles, and a small amount of vegetation, but it cannot directly label *sky*, *building*, *road*, *sidewalk*, or *signage*. We treat everything outside the named classes as `background`, and report it transparently.

If urban-scene parsing matters for your final project, swap in a Cityscapes-trained model (the `segformer-b0-finetuned-cityscapes-1024-1024` checkpoint on Hugging Face is a good starting point) and reuse the rest of the cell.

In [ ]:
# Module 6 — DeepLab v3 segmentation.
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import models, transforms

# Edit which segment image to segment.
SEGMENT_IMAGE_INDEX = 0

# Cache weights in Drive — set before loading the model.
os.environ["TORCH_HOME"] = os.path.join(OUTPUT_FOLDER, "models")
os.makedirs(os.environ["TORCH_HOME"], exist_ok=True)

# Skip re-loading if the model is already in memory.
if "deeplab_model" not in globals():
    print("Loading DeepLab v3 ResNet-50 (first run downloads ~170 MB)...")
    weights = models.segmentation.DeepLabV3_ResNet50_Weights.DEFAULT
    deeplab_model = models.segmentation.deeplabv3_resnet50(weights=weights)
    deeplab_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    deeplab_model.to(deeplab_device).eval()
    deeplab_categories = weights.meta["categories"]  # 21 Pascal VOC labels
    print(f"DeepLab loaded on {deeplab_device}.")
    print(f"Available raw classes: {deeplab_categories}")
else:
    print("DeepLab already loaded — reusing in-memory model.")

# Map raw VOC class indices to our urban design categories.
# Anything not listed below falls into 'unclassified (sky/building/road/etc.)'.
# This dict is intentionally readable: edit it to experiment.
CLASS_TO_URBAN = {
    "person":      "person",
    "car":         "vehicle",
    "bus":         "vehicle",
    "motorbike":   "vehicle",
    "bicycle":     "vehicle",
    "boat":        "vehicle",
    "train":       "vehicle",
    "aeroplane":   "vehicle",
    "pottedplant": "vegetation",
}
# Categories from the lab spec that DeepLab+VOC cannot resolve are tracked as
# 'unclassified' so the rest of the pipeline still gets a complete row.
URBAN_CATEGORIES = ["sky", "building", "vegetation", "road",
                    "sidewalk", "vehicle", "person", "signage", "unclassified"]

# Distinct colors per urban category for the overlay.
URBAN_COLORS = {
    "sky":          (135, 206, 250),
    "building":     (139,  69,  19),
    "vegetation":   ( 34, 139,  34),
    "road":         (105, 105, 105),
    "sidewalk":     (211, 211, 211),
    "vehicle":      (255, 140,   0),
    "person":       (220,  20,  60),
    "signage":      (255, 215,   0),
    "unclassified": ( 60,  60,  60),
}

deeplab_preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

def segment_image(pil_image):
    """Return a (H, W) numpy array of urban category names for each pixel."""
    inp = deeplab_preprocess(pil_image).unsqueeze(0).to(deeplab_device)
    with torch.no_grad():
        out = deeplab_model(inp)["out"][0]
    class_map = out.argmax(0).cpu().numpy()  # (H, W) of VOC class indices

    # Translate each VOC class index into our urban category string.
    urban_map = np.empty(class_map.shape, dtype=object)
    for idx, name in enumerate(deeplab_categories):
        urban = CLASS_TO_URBAN.get(name, "unclassified")
        urban_map[class_map == idx] = urban
    return urban_map

segment_dir = os.path.join(OUTPUT_FOLDER, "images")
segment_files = sorted(f for f in os.listdir(segment_dir) if f.startswith("seg_"))

if not segment_files:
    print("No segment images found. Run Module 3 first.")
elif SEGMENT_IMAGE_INDEX >= len(segment_files):
    print(f"SEGMENT_IMAGE_INDEX {SEGMENT_IMAGE_INDEX} out of range "
          f"(only {len(segment_files)} segment images).")
else:
    target_file = segment_files[SEGMENT_IMAGE_INDEX]
    img = Image.open(os.path.join(segment_dir, target_file)).convert("RGB")
    urban_map = segment_image(img)

    # Build the color overlay.
    h, w = urban_map.shape
    overlay_rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cat in URBAN_CATEGORIES:
        mask = urban_map == cat
        overlay_rgb[mask] = URBAN_COLORS[cat]

    # 50% opacity blend with the original.
    base = np.array(img.resize((w, h)))
    blended = (0.5 * base + 0.5 * overlay_rgb).astype(np.uint8)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(base)
    axes[0].set_title(f"Original — {target_file}")
    axes[0].axis("off")
    axes[1].imshow(blended)
    axes[1].set_title("Segmentation overlay (50% opacity)")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

    # Save the overlay.
    overlay_path = os.path.join(OUTPUT_FOLDER, "segment", "segmentation_overlay.jpg")
    Image.fromarray(blended).save(overlay_path, "JPEG", quality=92)
    print(f"Saved overlay: {overlay_path}")

    # Pixel counts per urban category.
    total_pixels = h * w
    rows = []
    for cat in URBAN_CATEGORIES:
        count = int((urban_map == cat).sum())
        if count > 0:
            rows.append({"category": cat,
                         "pixel_count": count,
                         "percentage": round(100 * count / total_pixels, 2)})
    df_seg = pd.DataFrame(rows).sort_values("pixel_count", ascending=False)
    csv_path = os.path.join(OUTPUT_FOLDER, "segment", "segmentation_data.csv")
    df_seg.to_csv(csv_path, index=False)
    print(f"Saved category counts: {csv_path}\n")
    # Styled inline table.
    try:
        from IPython.display import display
        display(df_seg.style.background_gradient(subset=["percentage"], cmap="Blues"))
    except Exception:
        print(df_seg.to_string(index=False))

**Reflection.** What does this pixel distribution tell you about the **pedestrian experience** of this block? Which category surprises you most — either by how much of the frame it occupies, or by how little? If `unclassified` dominates, that itself is a finding: it means the platform's default vision model has nothing to say about most of what is in front of you.

---

## Module 7 — Text and Signage Extraction with EasyOCR

EasyOCR runs optical character recognition on each image and returns every block of legible text along with a bounding box and a confidence score. We then classify each string with simple keyword heuristics into four buckets:

- **street_sign** — directional words, street suffixes (St, Ave, Blvd), cardinal directions.
- **regulatory** — STOP, NO, SPEED, LIMIT, ONE WAY, DO NOT, YIELD, PARKING.
- **wayfinding** — EXIT, ENTER, OPEN, HOURS, CLOSED.
- **business_name** — everything else above the confidence threshold.

> **Heads up:** the first time EasyOCR runs in a fresh session, it downloads ~100 MB of model weights and takes about a minute to initialize. Subsequent images are fast.

In [ ]:
# Module 7 — OCR and signage classification.
import os
import re
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import easyocr

# Skip re-initialization if the reader is already in memory.
if "ocr_reader" not in globals():
    print("Initializing EasyOCR (first run downloads ~100 MB)...")
    # gpu=True if the runtime has CUDA, otherwise EasyOCR falls back gracefully.
    import torch as _torch
    ocr_reader = easyocr.Reader(['en'], gpu=_torch.cuda.is_available())
    print("EasyOCR ready.")
else:
    print("EasyOCR already loaded — reusing in-memory reader.")

# Heuristic classifier. Returns a category label for a detected string.
STREET_SUFFIXES = {"ST", "AVE", "AVENUE", "BLVD", "BOULEVARD", "RD", "ROAD",
                   "STREET", "WAY", "PLACE", "PL", "DRIVE", "DR", "LANE", "LN"}
CARDINALS = {"NORTH", "SOUTH", "EAST", "WEST", "N", "S", "E", "W"}
REGULATORY_TOKENS = {"STOP", "NO", "SPEED", "LIMIT", "ONE WAY", "DO NOT",
                     "YIELD", "PARKING"}
WAYFINDING_TOKENS = {"EXIT", "ENTER", "OPEN", "HOURS", "CLOSED"}

def classify_text(text):
    upper = text.upper().strip()
    tokens = set(re.findall(r"[A-Z]+", upper))
    # Regulatory: substring match handles 'ONE WAY' and 'DO NOT'.
    for r in REGULATORY_TOKENS:
        if r in upper:
            return "regulatory"
    for w in WAYFINDING_TOKENS:
        if w in tokens:
            return "wayfinding"
    if tokens & STREET_SUFFIXES or tokens & CARDINALS:
        return "street_sign"
    return "business_name"

CATEGORY_COLORS = {
    "street_sign":   (  0, 200, 255),
    "regulatory":    (255,   0,   0),
    "wayfinding":    (  0, 200,   0),
    "business_name": (255, 200,   0),
}

CONFIDENCE_MIN = 0.3
MIN_TEXT_LEN   = 2

segment_dir = os.path.join(OUTPUT_FOLDER, "images")
segment_files = sorted(f for f in os.listdir(segment_dir) if f.startswith("seg_"))

if not segment_files:
    print("No segment images found. Run Module 3 first.")
else:
    all_rows = []
    for fname in tqdm(segment_files, desc="Reading signs"):
        path = os.path.join(segment_dir, fname)
        img_pil = Image.open(path).convert("RGB")
        img_np  = np.array(img_pil)

        try:
            results = ocr_reader.readtext(img_np)
        except Exception as e:
            print(f"OCR failed on {fname}: {e}")
            continue

        annotated = img_np.copy()
        for bbox, text, conf in results:
            if conf < CONFIDENCE_MIN or len(text.strip()) <= MIN_TEXT_LEN:
                continue
            category = classify_text(text)
            color = CATEGORY_COLORS[category]
            # bbox is a list of 4 (x,y) corners.
            pts = np.array(bbox, dtype=np.int32).reshape(-1, 1, 2)
            cv2.polylines(annotated, [pts], isClosed=True, color=color, thickness=2)
            cv2.putText(annotated, category[:3].upper(),
                        (int(bbox[0][0]), max(0, int(bbox[0][1]) - 6)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
            xs = [p[0] for p in bbox]; ys = [p[1] for p in bbox]
            all_rows.append({
                "image_filename": fname,
                "text": text.strip(),
                "classification": category,
                "confidence": round(float(conf), 3),
                "bbox_x": int(min(xs)),
                "bbox_y": int(min(ys)),
                "bbox_w": int(max(xs) - min(xs)),
                "bbox_h": int(max(ys) - min(ys)),
            })

        plt.figure(figsize=(8, 8))
        plt.imshow(annotated)
        plt.title(fname)
        plt.axis("off")
        plt.show()

    df_ocr = pd.DataFrame(all_rows)
    out_csv = os.path.join(OUTPUT_FOLDER, "ocr", "ocr_results.csv")
    df_ocr.to_csv(out_csv, index=False)
    print(f"\nTotal text detections: {len(df_ocr)}")
    print(f"Saved: {out_csv}\n")
    try:
        from IPython.display import display
        display(df_ocr)
    except Exception:
        print(df_ocr.to_string(index=False))

**Reflection.** What does the **signage ecology** of this street reveal about who it is designed for? A block dominated by regulatory text (NO PARKING, ONE WAY, SPEED LIMIT) is talking to drivers. A block dominated by business names and wayfinding is talking to pedestrians who already know what they are looking for. A block with mostly street signs and almost nothing else is talking to no one in particular — it is throughput infrastructure.

---

## Module 8 — Combined Structured Output and Visualization

Now we assemble one master row per Street View image, joining segmentation percentages, depth zone, and OCR counts to the lat/lng index from Module 3. We export it twice — once as a flat CSV for spreadsheets, once as a GeoJSON FeatureCollection for QGIS or MapLibre.

Then we make three small charts. None of them is the final story — they are sketches to help you find patterns to write about.

In [ ]:
# Module 8 — assemble master dataset, export CSV + GeoJSON, plot.
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Pull together everything from previous modules.
segment_index_path = os.path.join(OUTPUT_FOLDER, "images", "segment_index.csv")
if not os.path.exists(segment_index_path):
    print("segment_index.csv missing — run Module 3 first.")
    raise SystemExit
if "segment_image" not in globals():
    print("segment_image() not loaded — run Module 6 before this cell.")
    raise SystemExit
if "estimate_depth" not in globals():
    print("estimate_depth() not loaded — run Module 4 before this cell.")
    raise SystemExit

df_seg_idx = pd.read_csv(segment_index_path)

# Per-image segmentation: run DeepLab on every segment image so we get
# percentages for the full dataset (Module 6 only did one image inline).
print("Segmenting all segment images for the master table...")
per_image_seg = {}
segment_dir = os.path.join(OUTPUT_FOLDER, "images")
for fname in df_seg_idx["filename"]:
    try:
        img = Image.open(os.path.join(segment_dir, fname)).convert("RGB")
    except Exception as e:
        print(f"Could not open {fname}: {e}")
        continue
    urban_map = segment_image(img)  # function defined in Module 6
    h, w = urban_map.shape
    total = h * w
    pcts = {}
    for cat in URBAN_CATEGORIES:
        pcts[cat] = round(100 * float((urban_map == cat).sum()) / total, 2)
    per_image_seg[fname] = pcts

# Per-image depth zone using the same classifier as Module 4.
print("Computing depth zones for all segment images...")
per_image_depth = {}
for fname in df_seg_idx["filename"]:
    try:
        img = Image.open(os.path.join(segment_dir, fname)).convert("RGB")
    except Exception as e:
        print(f"Could not open {fname}: {e}")
        continue
    depth = estimate_depth(img)  # function from Module 4
    norm = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)
    m = float(norm.mean())
    if m >= 0.55:
        per_image_depth[fname] = "near"
    elif m >= 0.35:
        per_image_depth[fname] = "mid"
    else:
        per_image_depth[fname] = "far"

# OCR results loaded from disk so this cell does not redo OCR.
ocr_csv = os.path.join(OUTPUT_FOLDER, "ocr", "ocr_results.csv")
if os.path.exists(ocr_csv):
    df_ocr_all = pd.read_csv(ocr_csv)
else:
    df_ocr_all = pd.DataFrame(columns=["image_filename", "classification"])

ocr_counts = (df_ocr_all.groupby("image_filename").size()
              .rename("ocr_text_count").reset_index())
dominant = (df_ocr_all.groupby(["image_filename", "classification"]).size()
            .reset_index(name="n")
            .sort_values(["image_filename", "n"], ascending=[True, False])
            .drop_duplicates("image_filename")
            .rename(columns={"classification": "dominant_sign_category"})
            [["image_filename", "dominant_sign_category"]])

# Timeline year, if available — joined by lat/lng nearest match later if you
# extend the lab. For now we just leave capture_year blank for segment frames.
rows = []
for _, r in df_seg_idx.iterrows():
    fname = r["filename"]
    seg = per_image_seg.get(fname, {})
    rows.append({
        "image_filename": fname,
        "lat": r["lat"],
        "lng": r["lng"],
        "capture_year": None,
        "sky_pct":        seg.get("sky", 0.0),
        "building_pct":   seg.get("building", 0.0),
        "vegetation_pct": seg.get("vegetation", 0.0),
        "road_pct":       seg.get("road", 0.0),
        "sidewalk_pct":   seg.get("sidewalk", 0.0),
        "vehicle_pct":    seg.get("vehicle", 0.0),
        "person_pct":     seg.get("person", 0.0),
        "depth_zone":     per_image_depth.get(fname),
    })

df_master = pd.DataFrame(rows)
df_master = df_master.merge(ocr_counts, on="image_filename", how="left")
df_master = df_master.merge(dominant,   on="image_filename", how="left")
df_master["ocr_text_count"] = df_master["ocr_text_count"].fillna(0).astype(int)

# Export CSV.
csv_out = os.path.join(OUTPUT_FOLDER, "street_view_data.csv")
df_master.to_csv(csv_out, index=False)
print(f"Saved master CSV: {csv_out}")

# Export GeoJSON (one Point feature per image; CRS = EPSG:4326).
def _to_jsonable(v):
    """Coerce numpy scalars and NaN into JSON-friendly Python types."""
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except (TypeError, ValueError):
        pass
    if hasattr(v, "item"):  # numpy scalar
        return v.item()
    return v

features = []
for _, row in df_master.iterrows():
    props = {k: _to_jsonable(v) for k, v in row.items() if k not in ("lat", "lng")}
    features.append({
        "type": "Feature",
        "geometry": {"type": "Point",
                     "coordinates": [float(row["lng"]), float(row["lat"])]},
        "properties": props,
    })
geojson = {"type": "FeatureCollection",
           "crs": {"type": "name",
                   "properties": {"name": "urn:ogc:def:crs:EPSG::4326"}},
           "features": features}
geojson_out = os.path.join(OUTPUT_FOLDER, "street_view_data.geojson")
with open(geojson_out, "w") as f:
    json.dump(geojson, f, indent=2)
print(f"Saved GeoJSON: {geojson_out}")

# --- Plot 1: sky vs vegetation, sized by road, colored by depth zone.
zone_colors = {"near": "#b30000", "mid": "#ff7f00", "far": "#3182bd", None: "#999999"}
colors = [zone_colors.get(z, "#999999") for z in df_master["depth_zone"]]
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(df_master["sky_pct"], df_master["vegetation_pct"],
           s=df_master["road_pct"] * 10, c=colors, alpha=0.7,
           edgecolors="black", linewidths=0.5)
ax.set_xlabel("Sky %")
ax.set_ylabel("Vegetation %")
ax.set_title("Openness vs greenery — point size = road %, color = depth zone")
# Manual legend for depth zones.
import matplotlib.patches as mpatches
handles = [mpatches.Patch(color=c, label=z) for z, c in zone_colors.items()
           if z is not None]
ax.legend(handles=handles, title="Depth zone", loc="best")
plt.tight_layout()
plt.show()

# --- Plot 2: stacked bar of segmentation categories per image.
seg_cols = ["sky_pct", "building_pct", "vegetation_pct", "road_pct",
            "sidewalk_pct", "vehicle_pct", "person_pct"]
fig, ax = plt.subplots(figsize=(10, 0.5 + 0.4 * len(df_master)))
left = np.zeros(len(df_master))
palette = ["#87CEFA", "#8B4513", "#228B22", "#696969",
           "#D3D3D3", "#FF8C00", "#DC143C"]
for col, color in zip(seg_cols, palette):
    ax.barh(df_master["image_filename"], df_master[col],
            left=left, color=color, label=col.replace("_pct", ""))
    left += df_master[col].values
ax.set_xlabel("Percent of frame")
ax.set_title("Segmentation breakdown per image")
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
plt.tight_layout()
plt.show()

# --- Plot 3: OCR text count per image, colored by dominant sign category.
dom_palette = {"street_sign": "#00C8FF", "regulatory": "#FF0000",
               "wayfinding": "#00C800", "business_name": "#FFC800",
               None: "#BBBBBB"}
fig, ax = plt.subplots(figsize=(10, 4))
bar_colors = [dom_palette.get(c, "#BBBBBB") for c in df_master["dominant_sign_category"]]
ax.bar(df_master["image_filename"], df_master["ocr_text_count"],
       color=bar_colors, edgecolor="black", linewidth=0.5)
ax.set_xticks(range(len(df_master)))
ax.set_xticklabels(df_master["image_filename"], rotation=60, ha="right", fontsize=8)
ax.set_ylabel("Text detections")
ax.set_title("Signage volume per frame (color = dominant sign category)")
plt.tight_layout()
plt.show()

# Preview the master DataFrame inline.
try:
    from IPython.display import display
    display(df_master)
except Exception:
    print(df_master.to_string(index=False))

---

## Closing reflection

Pick the three prompts most useful to your final project and answer them in writing — in this notebook, your field journal, or your studio narrative.

1. **Which segment images have the highest enclosure?** How does that correlate with vegetation or sky coverage in the segmentation table? Where does your embodied memory of the street agree with what the model "saw," and where does it disagree?

2. **Where does signage cluster along the street?** What might explain that pattern — a transit node, a retail corridor, a school zone, the edge of a regulatory boundary? Which sign category is doing the most narrative work in this block?

3. **If you were redesigning one block of this street, which data point would you prioritize and why?** Be specific. "Reduce vehicle %" is a goal; "narrow the curb-to-curb width by 8 feet to bring vehicle % from 35% to ~22% and let sidewalk_pct rise correspondingly" is a design move with a measurable test.